# Offline Ablation — Step 3 (A1, A2, A3)

Based on the **original Step 3 baseline notebook**. Outputs are intermediate 400-record repositories for subsequent Step 4 refinement (not final online repositories).

- **A1**: diversity by stance on `*_clean_pro_anti_A1.csv` from A1 Step 2 (when FULL is ready). No artificial padding if fewer than 400 records survive.
- **A2**: diversity **without stance labels or stance quotas** on original `*_top8000_before_gpt.csv`. Exactly up to 400 rows. No GPT stance labeling here.
- **A3**: **no diversity**; random 200 pro + 200 anti from original baseline `*_top8000_all_labeled.csv`, after original `pro/anti` + confidence ≥0.60 filter.

**A2 input is NOT the A1 random FULL sample.** A2 uses the original topic-aware Top-8000. Select `RUN_VARIANTS` to avoid unnecessary work. TEST and FULL outputs are isolated.

In [ ]:
!pip -q install pandas numpy tqdm sentence-transformers

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

In [ ]:
# CONFIG — update this one directory to your actual Drive folder.
BASE = Path('/content')
INPUT_DIR_A1 = BASE / 'stage2/FULL/A1'
INPUT_DIR_A2_A3 = BASE / 'baseline_results'
OUTPUT_DIR = BASE / 'stage3'
RUN_MODE = 'FULL'             # TEST / FULL
RUN_VARIANTS = ['A1']   # add 'A1' once A1 Step 2 FULL results are available
TEST_N = 600                  # small diagnostic input per topic in TEST; FULL uses all
CHECKPOINT_EVERY = 5   # Save diversity selection every 5 chosen rows
RANDOM_SEED = 42
K_TOTAL_PER_TOPIC = 400
TEXT_COL = 'comment_text'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
LAMBDA_DIVERSITY = 0.45
CONFIDENCE_THRESHOLD = 0.60

# Put files in INPUT_DIR with these names (rename only copies, not originals).
FILES = {
    'A1': {'blm':INPUT_DIR_A1/'blm_clean_pro_anti_A1.csv', 'guns':INPUT_DIR_A1/'guns_clean_pro_anti_A1.csv', 'samesex':INPUT_DIR_A1/'samesex_clean_pro_anti_A1.csv'},
    'A2': {'blm':INPUT_DIR_A2_A3/'blm_top8000_before_gpt.csv', 'guns':INPUT_DIR_A2_A3/'guns_top8000_before_gpt.csv', 'samesex':INPUT_DIR_A2_A3/'samesex_top8000_before_gpt.csv'},
    'A3': {'blm':INPUT_DIR_A2_A3/'blm_top8000_all_labeled.csv', 'guns':INPUT_DIR_A2_A3/'guns_top8000_all_labeled.csv', 'samesex':INPUT_DIR_A2_A3/'samesex_top8000_all_labeled.csv'},
}
assert RUN_MODE in ('TEST','FULL')
assert set(RUN_VARIANTS) <= {'A1','A2','A3'}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Run:',RUN_MODE,RUN_VARIANTS)

In [ ]:

def compute_balanced_quotas(df, k_total=400, stance_col="stance_label"):
    counts = df[stance_col].value_counts().to_dict()

    n_pro = int(counts.get("pro", 0))
    n_anti = int(counts.get("anti", 0))

    half = k_total // 2  # 200 per stance

    pro_quota = min(half, n_pro)
    anti_quota = min(half, n_anti)

    total_selected = pro_quota + anti_quota

    if total_selected < k_total:
        print(
            f"WARNING: only {total_selected} rows can be selected "
            f"under the stance quotas "
            f"(pro={pro_quota}, anti={anti_quota})."
        )

    return {
        "pro": pro_quota,
        "anti": anti_quota
    }


In [ ]:
# ============================================================
# DIVERSE + REPRESENTATIVE SELECTION WITHIN ONE STANCE
# ============================================================

def select_diverse_representative_from_embeddings(
    embeddings,
    k,
    lambda_diversity=0.45,
    seed=42,
    checkpoint_path=None,
    checkpoint_meta=None
):
    """
    Select k rows from a clean stance-specific dataset.

    Since your clean stance subsets are only a few thousand rows,
    this version uses all rows as the candidate pool and all rows for coverage.
    No CANDIDATE_POOL_SIZE, COVERAGE_SAMPLE_SIZE, or EVAL_SIZE_PER_STEP is needed.
    """

    rng = np.random.default_rng(seed)

    n = embeddings.shape[0]

    if k >= n:
        return list(range(n))

    emb = np.asarray(embeddings, dtype=np.float32)

    # full similarity matrix: [all rows x all candidates]
    sim_matrix = emb @ emb.T

    selected = []

    current_coverage = np.full(n, -1.0, dtype=np.float32)
    selected_mask = np.zeros(n, dtype=bool)

    # start with the comment closest to the centroid
    center = emb.mean(axis=0)
    center = center / (np.linalg.norm(center) + 1e-12)

    centroid_sims = emb @ center
    first = int(np.argmax(centroid_sims))

    selected.append(first)
    selected_mask[first] = True
    current_coverage = np.maximum(current_coverage, sim_matrix[:, first])

    if checkpoint_path is not None and Path(checkpoint_path).exists():
        state = json.loads(Path(checkpoint_path).read_text(encoding='utf-8'))
        if state.get('meta') != checkpoint_meta:
            raise ValueError(f'Checkpoint does not match input/config: {checkpoint_path}')
        saved = state.get('selected')
        if not isinstance(saved,list) or not saved or saved[0]!=first or len(saved)!=len(set(saved)) or len(saved)>k or any(not isinstance(i,int) or i<0 or i>=n for i in saved):
            raise ValueError(f'Invalid selection checkpoint: {checkpoint_path}')
        selected = saved
        selected_mask[:] = False
        selected_mask[selected] = True
        current_coverage = np.full(n,-1.0,dtype=np.float32)
        for idx in selected:
            current_coverage = np.maximum(current_coverage, sim_matrix[:, idx])
        print(f'[RESUME] {checkpoint_path}: {len(selected)}/{k}')

    for _ in tqdm(range(1, k), desc=f"Selecting {k}"):
        available = np.where(~selected_mask)[0]

        candidate_sims_to_dataset = sim_matrix[:, available]
        new_coverage = np.maximum(current_coverage[:, None], candidate_sims_to_dataset)
        coverage_gains = np.sum(new_coverage - current_coverage[:, None], axis=0)

        selected_array = np.array(selected, dtype=int)
        candidate_to_selected_sims = sim_matrix[available][:, selected_array]
        max_sim_to_selected = np.max(candidate_to_selected_sims, axis=1)
        diversity_gains = 1.0 - max_sim_to_selected

        scores = coverage_gains + lambda_diversity * n * diversity_gains

        best_pos = int(np.argmax(scores))
        best_id = int(available[best_pos])

        selected.append(best_id)
        selected_mask[best_id] = True
        current_coverage = np.maximum(current_coverage, sim_matrix[:, best_id])
        if checkpoint_path is not None and (len(selected)%CHECKPOINT_EVERY==0 or len(selected)==k):
            atomic_json(checkpoint_path, {'meta':checkpoint_meta,'selected':selected})

    return selected

In [ ]:
import os, json, hashlib

def atomic_json(path, obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(str(path)+'.tmp')
    with open(tmp,'w',encoding='utf-8') as f: json.dump(obj,f,ensure_ascii=False)
    os.replace(tmp,path)

def atomic_csv(df,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(str(path)+'.tmp')
    df.to_csv(tmp,index=False,encoding='utf-8-sig')
    os.replace(tmp,path)

def dataframe_signature(df):
    h=hashlib.sha256()
    for row in df.astype(str).itertuples(index=False,name=None):
        h.update(json.dumps(row,ensure_ascii=False).encode('utf-8'));h.update(b'\n')
    return h.hexdigest()

# Load embedding model only when a diversity variant is requested.
model = SentenceTransformer(MODEL_NAME) if set(RUN_VARIANTS) & {'A1','A2'} else None

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Current model device:", model.device)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Model moved to:", model.device)

In [ ]:
def load_topic(variant, topic):
    path=FILES[variant][topic]
    if not path.exists():
        raise FileNotFoundError(f'Missing {variant}/{topic}: {path}')
    df=pd.read_csv(path, encoding='utf-8-sig')
    if TEXT_COL not in df.columns:
        raise ValueError(f'Missing {TEXT_COL} in {path}')
    if 'comment_id' in df.columns and df.comment_id.duplicated().any():
        print(f'WARNING: duplicate comment_id in {path.name}:',int(df.comment_id.duplicated().sum()))
    return df

def select_diversity(df, k, variant, topic, group='all', seed=RANDOM_SEED):
    if not len(df): return df.copy()
    sig=dataframe_signature(df)
    checkpoint_dir=OUTPUT_DIR/RUN_MODE/variant/'checkpoints'/topic/group
    checkpoint_dir.mkdir(parents=True,exist_ok=True)
    emb_path=checkpoint_dir/'embeddings.npy'
    emb_meta_path=checkpoint_dir/'embeddings_meta.json'
    emb_meta={'signature':sig,'model':MODEL_NAME,'rows':len(df),'normalized':True}
    if emb_path.exists() or emb_meta_path.exists():
        if not emb_path.exists() or not emb_meta_path.exists() or json.loads(emb_meta_path.read_text())!=emb_meta:
            raise ValueError(f'Embedding cache mismatch: {checkpoint_dir}')
        emb=np.load(emb_path)
        if emb.shape[0]!=len(df) or not np.isfinite(emb).all():
            raise ValueError(f'Invalid embeddings: {emb_path}')
        print(f'[RESUME] embeddings {variant}/{topic}/{group}: {len(df)}')
    else:
        texts=df[TEXT_COL].fillna('').astype(str).tolist()
        emb=model.encode(texts,batch_size=256,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)
        tmp=Path(str(emb_path)+'.tmp')
        with open(tmp,'wb') as f: np.save(f,emb)
        os.replace(tmp,emb_path)
        atomic_json(emb_meta_path,emb_meta)
    meta={**emb_meta,'k':min(k,len(df)),'lambda':LAMBDA_DIVERSITY,'seed':seed,'algorithm':'baseline_full_matrix_greedy_v1'}
    idx=select_diverse_representative_from_embeddings(
        embeddings=emb,k=min(k,len(df)),lambda_diversity=LAMBDA_DIVERSITY,seed=seed,
        checkpoint_path=checkpoint_dir/'selection.json',checkpoint_meta=meta)
    return df.iloc[idx].copy()

def process_a1(df, topic):
    # Original baseline diversity algorithm and original stance-based quota logic.
    df=df.copy()
    df['stance_label']=df['stance_label'].astype(str).str.lower().str.strip()
    df=df[df['stance_label'].isin(['pro','anti']) & (pd.to_numeric(df['confidence'],errors='coerce')>=CONFIDENCE_THRESHOLD)].copy().reset_index(drop=True)
    if RUN_MODE=='TEST': df=df.groupby('stance_label',group_keys=False).head(TEST_N//2).reset_index(drop=True)
    quotas=compute_balanced_quotas(df,k_total=K_TOTAL_PER_TOPIC,stance_col='stance_label')
    parts=[]
    for stance,quota in quotas.items():
        sub=df[df.stance_label==stance].copy().reset_index(drop=True)
        if quota==0:continue
        chosen=select_diversity(sub,quota,variant="A1",topic=topic,group=stance)
        chosen['selection_stance_quota']=quota
        chosen['selection_stage']='balanced_diverse_selection'
        parts.append(chosen)
    if not parts: return df.iloc[:0].copy()
    result=pd.concat(parts,ignore_index=True).sample(frac=1,random_state=RANDOM_SEED).reset_index(drop=True)
    result['topic_selection']=topic
    result['selection_rank']=np.arange(1,len(result)+1)
    return result

def process_a2(df,topic):
    # Exactly the same embedding and diversity scoring function, but no stance splitting or quotas.
    df=df.reset_index(drop=True).copy()
    if RUN_MODE=='TEST':df=df.head(TEST_N).copy()
    result=select_diversity(df,K_TOTAL_PER_TOPIC,variant="A2",topic=topic).reset_index(drop=True)
    result['selection_stage']='diverse_selection_no_stance'
    result['topic_selection']=topic
    result['selection_rank']=np.arange(1,len(result)+1)
    return result

def process_a3(df,topic):
    # Original Step 2 filtering; replace diversity with seeded random sampling.
    df=df.copy()
    df['stance_label']=df['stance_label'].astype(str).str.lower().str.strip()
    df['confidence']=pd.to_numeric(df['confidence'],errors='coerce')
    df=df[df.stance_label.isin(['pro','anti']) & (df.confidence>=CONFIDENCE_THRESHOLD)].reset_index(drop=True)
    if RUN_MODE=='TEST':
        df=pd.concat([df[df.stance_label==s].head(TEST_N) for s in ('pro','anti')],ignore_index=True)
    parts=[]
    for stance,seed in [('pro',RANDOM_SEED),('anti',RANDOM_SEED+1)]:
        sub=df[df.stance_label==stance]
        k=min(200,len(sub))
        if RUN_MODE=='FULL' and k<200:
            raise ValueError(f'A3 {topic}: only {k} {stance} records; cannot sample 200')
        chosen=sub.sample(n=k,replace=False,random_state=seed).copy()
        chosen['selection_stance_quota']=k
        chosen['selection_stage']='random_selection_no_diversity'
        parts.append(chosen)
    result=pd.concat(parts,ignore_index=True).sample(frac=1,random_state=RANDOM_SEED).reset_index(drop=True)
    result['topic_selection']=topic
    result['selection_rank']=np.arange(1,len(result)+1)
    return result


In [ ]:
# Run with persistent per-topic outputs; completed outputs are reused only if input/config matches.
results=[]
for variant in RUN_VARIANTS:
    for topic in ('blm','guns','samesex'):
        print('\n',variant,topic)
        df=load_topic(variant,topic)
        dest=OUTPUT_DIR/RUN_MODE/variant
        dest.mkdir(parents=True,exist_ok=True)
        manifest=dest/f'{topic}_{variant}_manifest.json'
        input_sig=dataframe_signature(df)
        expected={'input_sha256':input_sig,'mode':RUN_MODE,'variant':variant,'topic':topic,
                  'test_n':TEST_N if RUN_MODE=='TEST' else None,'k':K_TOTAL_PER_TOPIC,
                  'seed':RANDOM_SEED,'model':MODEL_NAME,'lambda':LAMBDA_DIVERSITY,
                  'confidence_threshold':CONFIDENCE_THRESHOLD}
        if manifest.exists():
            saved=json.loads(manifest.read_text(encoding='utf-8'))
            if saved.get('config')!=expected:raise ValueError(f'Existing output config mismatch: {manifest}')
            path=Path(saved['output'])
            if not path.exists():raise FileNotFoundError(path)
            selected=pd.read_csv(path,encoding='utf-8-sig')
            if len(selected)!=saved['rows']:raise ValueError(f'Output row count mismatch: {path}')
            print('[RESUME] Verified completed output:',path)
        else:
            if variant=='A1':selected=process_a1(df,topic)
            elif variant=='A2':selected=process_a2(df,topic)
            else:selected=process_a3(df,topic)
            path=dest/f'{topic}_{variant}_selected_{len(selected)}_step3.csv'
            atomic_csv(selected,path)
            atomic_json(manifest,{'config':expected,'output':str(path),'rows':len(selected)})
            print('Saved:',path)
        counts=selected['stance_label'].value_counts().to_dict() if 'stance_label' in selected else {'unlabeled':len(selected)}
        print('Rows:',len(selected),'counts:',counts)
        results.append({'variant':variant,'topic':topic,'rows':len(selected),'stance_counts':str(counts),'output':str(path)})
summary=pd.DataFrame(results)
atomic_csv(summary,OUTPUT_DIR/RUN_MODE/'step3_summary.csv')
display(summary)


## Before Step 4

For FULL: **A2** and **A3** should each produce 400 records per topic; **A1** can have fewer if insufficient candidates pass stance filtering. These are **intermediate outputs**, not the final 300-record repositories. Step 4 must use the original six-criterion GPT refinement: stance-balanced 150+150 for A1/A3 where possible; for A2, adapt the prompt to not assume labels and select the 300 highest scoring records without quotas.

**Performance note:** Original Step 3 computes a full `n × n` similarity matrix and greedily selects 400 items. For A2 on all 8,000 candidates this can be very slow and memory intensive. This notebook deliberately preserves that original scoring algorithm rather than silently replacing it with an approximation. Test first.